In [1]:
from PIL import Image, ImageDraw, ImageFont

def extract_image_info(image_path):
    # Open the image using Pillow
    img = Image.open(image_path)

    # Get image size (in pixels)
    width, height = img.size
    print(f"Image Resolution: {width} x {height} pixels")

    # Get DPI (dots per inch) if available in image metadata
    dpi = img.info.get('dpi', None)
    if dpi:
        print(f"Image DPI: {dpi}")
    else:
        print("DPI not available in the image metadata.")
    
    return img, width, height, dpi

def calculate_scaling_factor(dpi, width):
    if dpi:
        # Assuming DPI is (x, y) - we use x DPI for calculation
        dpi_x = dpi[0]
        # Convert pixel to inches, then to mm
        pixels_per_inch = dpi_x
        mm_per_pixel = 25.4 / pixels_per_inch  # 25.4mm in 1 inch
        print(f"Scaling Factor: {mm_per_pixel} mm per pixel")
        return mm_per_pixel
    else:
        print("DPI is not available, scaling will be based on assumption.")
        return None

def add_grid_to_image(img, width, height, scaling_factor, grid_size_mm=10):
    # Convert grid size in mm to pixels
    if scaling_factor:
        grid_size_pixels = grid_size_mm / scaling_factor
        print(f"Grid size: {grid_size_mm} mm = {grid_size_pixels:.2f} pixels")
    else:
        grid_size_pixels = 50  # Default size in pixels if scaling factor is unavailable
        print(f"Default grid size: {grid_size_pixels} pixels")

    # Create a drawing object
    draw = ImageDraw.Draw(img)
    
    # Draw vertical lines at every `grid_size_pixels` pixels
    for x in range(0, width, int(grid_size_pixels)):
        draw.line([(x, 0), (x, height)], fill="red", width=2)
    
    # Draw horizontal lines at every `grid_size_pixels` pixels
    for y in range(0, height, int(grid_size_pixels)):
        draw.line([(0, y), (width, y)], fill="red", width=2)

    # Add ruler on the right side of the image
    add_ruler_to_image(draw, width, height, scaling_factor, grid_size_mm)

    # Display the image with the grid
    img.show()

    return img

def add_ruler_to_image(draw, width, height, scaling_factor, grid_size_mm):
    # Add a ruler-like scale on the right side of the image
    margin = 10  # Margin from the right edge
    ruler_x_position = width - margin  # Right edge of the image

    # Load a font (use default if no specific font is available)
    try:
        font = ImageFont.load_default()
    except IOError:
        print("Error loading default font.")
        font = ImageFont.load_default()

    # Calculate the vertical center of the image
    ruler_label_y_position = height // 2 - 10  # Centered vertically, adjust if needed

    # Draw the ruler label in the middle vertically
    draw.text((ruler_x_position - 60, ruler_label_y_position), "Ruler (mm)", font=font, fill="blue")

    # Draw the ruler scale on the right side
    for y in range(0, height, int(grid_size_mm / scaling_factor)):
        # Label with mm values
        label = f"{(y * scaling_factor) / 25.4:.1f} mm"
        
        # Draw a small line for each grid mark (on the right)
        draw.line([(ruler_x_position, y), (ruler_x_position - 5, y)], fill="blue", width=2)
        
        # Add the label text next to the line
        label_y_position = y - 10  # Adjust label position slightly for clarity
        draw.text((ruler_x_position - 70, label_y_position), label, font=font, fill="blue")

# Example usage
image_path = "trials/diagnostics-14-02338-g001.png"  # Replace with your image file path
img, width, height, dpi = extract_image_info(image_path)

# Calculate the scaling factor based on DPI (if available)
scaling_factor = calculate_scaling_factor(dpi, width)

# Add a grid with a cell size of 10mm and a ruler on the right
img_with_grid = add_grid_to_image(img, width, height, scaling_factor, grid_size_mm=10)

# Optionally, save the new image with the grid
img_with_grid.save("image_with_grid_and_ruler.png")


FileNotFoundError: [Errno 2] No such file or directory: 'trials/diagnostics-14-02338-g001.png'

In [28]:
import cv2
import numpy as np
from tkinter import Tk, filedialog

# Parameters for grid
grid_size_cm = 1  # grid size in centimeters (1 cm)
pixels_per_cm = 10  # Example scale: 10 pixels per centimeter

# Global variables for scaling and translation
scale_factor = 1.0  # Initial scale factor
x_offset = 0  # Initial x offset
y_offset = 0  # Initial y offset

# Mouse callback function for dragging and zooming
def mouse_callback(event, x, y, flags, param):
    global x_offset, y_offset, scale_factor, image, orig_image

    if event == cv2.EVENT_LBUTTONDOWN:
        # Store initial position when the left mouse button is pressed
        x_offset, y_offset = x, y
    elif event == cv2.EVENT_MOUSEMOVE and flags == cv2.EVENT_FLAG_LBUTTON:
        # Move the image based on mouse movement
        dx = x - x_offset
        dy = y - y_offset
        image = orig_image.copy()  # Reload the original image
        image = cv2.resize(image, (int(image.shape[1] * scale_factor), int(image.shape[0] * scale_factor)))
        image = np.roll(image, dx, axis=1)  # Adjust x offset
        image = np.roll(image, dy, axis=0)  # Adjust y offset
        cv2.imshow("Image with Ruler", image)

    elif event == cv2.EVENT_MOUSEWHEEL:
        # Zoom in or out using the mouse scroll wheel
        if flags > 0:  # Scroll up (zoom in)
            scale_factor *= 1.1
        else:  # Scroll down (zoom out)
            scale_factor /= 1.1
        image_resized = cv2.resize(orig_image, (int(orig_image.shape[1] * scale_factor), int(orig_image.shape[0] * scale_factor)))
        cv2.imshow("Image with Ruler", image_resized)

# Function to draw the grid (ruler) on the image
def draw_grid(image):
    height, width, _ = image.shape

    # Draw vertical lines at every `grid_size_cm * pixels_per_cm` pixels
    for x in range(0, width, grid_size_cm * pixels_per_cm):
        cv2.line(image, (x, 0), (x, height), (0, 255, 0), 1)  # Green vertical lines
    
    # Draw horizontal lines at every `grid_size_cm * pixels_per_cm` pixels
    for y in range(0, height, grid_size_cm * pixels_per_cm):
        cv2.line(image, (0, y), (width, y), (0, 255, 0), 1)  # Green horizontal lines

    return image

# Function to open file dialog and ask user to select an image
def upload_image():
    # Use Tkinter file dialog to upload the image
    Tk().withdraw()  # Hide the root window
    file_path = filedialog.askopenfilename(title="Select an X-ray Image", filetypes=[("Image files", "*.png;*.jpg;*.jpeg;*.tiff")])
    
    if file_path:
        return cv2.imread(file_path)
    else:
        print("No image selected.")
        return None

# Main function
def main():
    # Ask the user to upload an image
    image = upload_image()
    
    if image is None:
        return
    
    # Store the original image for resizing purposes
    global orig_image
    orig_image = image.copy()
    
    # Draw the grid on the image
    image_with_grid = draw_grid(image)
    
    # Display the image with the grid overlay
    cv2.imshow("Image with Ruler", image_with_grid)

    # Set the mouse callback function for image interaction
    cv2.setMouseCallback("Image with Ruler", mouse_callback)

    # Wait for a key press to close the window
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Run the program
if __name__ == "__main__":
    main()


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'
